In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

#Load the dataset

In [ ]:
#Load Dataset
(ds_train,ds_val,ds_test),info=tfds.load(
    'tf_flowers',
    split=['train[:70%]','train[70%:85%]','train[85%:]'],
    as_supervised=True,
    with_info=True
)
NUM_CLASSES=info.features['label'].num_classes
CLASS_NAMES=info.features['label'].names



In [ ]:
CLASS_NAMES

#Preprocessing the dataset

In [ ]:
IMG_SIZE=224
BATCH_SIZE=32

def preprocess(image,label):
  image=tf.image.resize(image,(IMG_SIZE,IMG_SIZE))
  image=tf.keras.applications.mobilenet_v2.preprocess_input(image)
  return image,label

train_ds=ds_train.map(preprocess).shuffle(1000).batch(BATCH_SIZE)
val_ds=ds_val.map(preprocess).batch(BATCH_SIZE)
test_ds=ds_test.map(preprocess).batch(BATCH_SIZE)

#Data Augementation

In [ ]:
data_augmentation=tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)

])

#Creating the basemodel

In [ ]:
base_model=tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE,IMG_SIZE,3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable=False

#Build the sequential model

In [ ]:
from tensorflow.keras import models
from tensorflow.keras.layers import Dense,Flatten,Dropout,GlobalAveragePooling2D



In [ ]:
model=models.Sequential()
#Adding augmentation
model.add(data_augmentation)
#Adding Base model
model.add(base_model)

#Add classification head
model.add(GlobalAveragePooling2D())
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(NUM_CLASSES,activation='softmax'))
#

In [ ]:
model.build(input_shape=(None, 224, 224, 3))
model.summary()

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [ ]:
history=model.fit(
    train_ds,
    epochs=50,
    validation_data=val_ds
)

#Fine tuning the model

🎯 Simple Summary

👉 Phase 1: Train only new layers while freezing pretrained model

👉 Phase 2: Unfreeze top layers and fine-tune with a low learning rate

In [ ]:
base_model.trainable=True

for layers in base_model.layers[:-30]:
  layers.trainable=False

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(1e-5),
    metrics=['accuracy']
)

In [ ]:
history2 = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds
)

#Evaluating

In [ ]:
loss,acc=model.evaluate(test_ds)
print("Model Accuracy: ",acc )

In [ ]:
def show_predictions(data):
  images, labels = next(iter(data))
  preds=model.predict(images)
  pred_labels=np.argmax(preds,axis=1)
  print(preds)
  print(pred_labels)

show_predictions(test_ds)

In [ ]:
def show_predictions_p(dataset):
    images, labels = next(iter(dataset))
    preds = model.predict(images)
    pred_labels = np.argmax(preds, axis=1)

    plt.figure(figsize=(10,10))
    for i in range(9):
        plt.subplot(3,3,i+1)
        plt.imshow((images[i] + 1) / 2)  # fix display
        plt.title(
            f"Actual: {CLASS_NAMES[labels[i]]}\n"
            f"Pred: {CLASS_NAMES[pred_labels[i]]}"
        )

        plt.axis('off')
    plt.show()

show_predictions_p(test_ds)

In [ ]:
model.save("flower_model.h5")